## Notebook summary

| Item | Details |
| --- | --- |
| Purpose | 03 - Locked YOLO ROI Test Evaluation |
| Model / workflow | SE-ResNeXt-50 32x4d |
| Input | 384x384 YOLO ROI test split |
| Loss | Cross-Entropy (CE) checkpoint |
| Training / pipeline | Evaluation only |
| Result | (filled in after the run) |

# 03 - Locked YOLO ROI Test Evaluation (SE-ResNeXt-50 32x4d)

Evaluates the Notebook 02 checkpoint **once** on the untouched YOLO ROI test split. It does not
train, tune, or overwrite anything.

Outputs written to the run directory:

- `test_metrics.json` and `test_predictions.csv`
- `test_confusion_matrix.png`
- Grad-CAM grids for correct cases, and predicted-vs-true CAM pairs for errors
- `report_row.json`, formatted for pasting into `docs/report/report.csv`

Do not use these metrics to pick a different epoch or preprocessing setting. If you do, the split
stops being a held-out test set.

In [ ]:
!pip -q install "timm>=1.0" "h5py>=3.9"

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import json
import random
from datetime import datetime, timezone
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import timm
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import (
    accuracy_score, average_precision_score, classification_report,
    cohen_kappa_score, confusion_matrix, precision_recall_fscore_support,
    roc_auc_score,
)
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from tqdm.auto import tqdm

# ---- Focal CORN helpers (inlined for self-containment) -----------------
NUM_CLASSES = 5
NUM_TASKS = NUM_CLASSES - 1
TASK_WEIGHTS = (1.0, 1.2, 2.0, 3.5)
FOCAL_GAMMA = 2.0
FOCAL_ALPHA = 0.25
LABEL_SMOOTHING = 0.10

def corn_loss(logits, y_train, num_classes=NUM_CLASSES, task_weights=TASK_WEIGHTS):
    loss = 0.0
    for k in range(num_classes - 1):
        mask = y_train >= k
        if not mask.any():
            continue
        logits_k = logits[mask, k]
        targets_k = (y_train[mask] > k).float()
        targets_k = targets_k * (1 - LABEL_SMOOTHING) + (1 - targets_k) * LABEL_SMOOTHING
        w_k = task_weights[k] if k < len(task_weights) else 1.0
        loss = loss + w_k * F.binary_cross_entropy_with_logits(logits_k, targets_k)
    return loss / (num_classes - 1)

def focal_corn_loss(logits, y_train, num_classes=NUM_CLASSES, gamma=FOCAL_GAMMA, alpha=FOCAL_ALPHA):
    loss = 0.0
    for k in range(num_classes - 1):
        mask = y_train >= k
        if not mask.any():
            continue
        logits_k = logits[mask, k]
        targets_k = (y_train[mask] > k).float()
        targets_k = targets_k * (1 - LABEL_SMOOTHING) + (1 - targets_k) * LABEL_SMOOTHING
        bce = F.binary_cross_entropy_with_logits(logits_k, targets_k, reduction="none")
        p = torch.sigmoid(logits_k)
        p_t = p * targets_k + (1 - p) * (1 - targets_k)
        focal_weight = alpha * (1 - p_t) ** gamma
        loss = loss + (focal_weight * bce).mean()
    return loss / (num_classes - 1)

def corn_probas(logits):
    cond_probas = torch.sigmoid(logits)
    batch_size = logits.size(0)
    num_classes = logits.size(1) + 1
    probas = torch.zeros(batch_size, num_classes, device=logits.device)
    cumprod = torch.cumprod(cond_probas, dim=1)
    probas[:, 0] = 1.0 - cond_probas[:, 0]
    for i in range(1, num_classes - 1):
        probas[:, i] = cumprod[:, i - 1] * (1.0 - cond_probas[:, i])
    probas[:, -1] = cumprod[:, -1]
    return probas

def corn_label_from_logits(logits):
    return torch.argmax(corn_probas(logits), dim=1)

## Configuration

`CHECKPOINT_PATH` resolves from the newest Notebook 02 pointer. Override it manually to evaluate a
specific historical checkpoint instead.

In [ ]:
# ---- reproducibility -------------------------------------------------------
SEED = 42

# ---- data ------------------------------------------------------------------
INPUT_SIZE = 384
BATCH_SIZE = 48
NUM_WORKERS = 2
CASES_PER_GRADE = 5
PRETRAINED = False

# ---- paths -----------------------------------------------------------------
ROI_TEST_ROOT = Path(
    "/content/drive/MyDrive/Datasets/KneeXrayData_Mendeley_v1/"
    "derived/densenet121_yolo_square_roi_trainvaltest_v2/test"
)
PAIRED_MODEL_ROOT = Path("/content/drive/MyDrive/Models/seresnext50_32x4d_paired_roi_focal_corn")
OUTPUT_ROOT = Path("/content/drive/MyDrive/Models/seresnext50_32x4d_evaluation_focal_corn")

# Set this to a Path to evaluate a specific checkpoint; None picks the newest Notebook 02 run.
CHECKPOINT_OVERRIDE = None

if CHECKPOINT_OVERRIDE is not None:
    CHECKPOINT_PATH = Path(CHECKPOINT_OVERRIDE)
else:
    pointers = sorted(
        PAIRED_MODEL_ROOT.glob("*/SELECTED_CHECKPOINT.txt"),
        key=lambda path: path.stat().st_mtime, reverse=True,
    )
    if not pointers:
        raise FileNotFoundError(
            f"No Notebook 02 pointer under {PAIRED_MODEL_ROOT}. Run 02_train_paired_roi.ipynb first."
        )
    CHECKPOINT_PATH = Path(pointers[0].read_text().strip())

RUN_TIMESTAMP = datetime.now(timezone.utc).strftime("%Y-%m-%d_%H-%M-%S_%f_UTC")
OUTPUT_DIR = OUTPUT_ROOT / RUN_TIMESTAMP

for required in (ROI_TEST_ROOT, CHECKPOINT_PATH):
    if not required.exists():
        raise FileNotFoundError(required)
OUTPUT_DIR.mkdir(parents=True, exist_ok=False)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:    ", DEVICE)
print("Checkpoint:", CHECKPOINT_PATH)
print("Test split:", ROI_TEST_ROOT)
print("Output dir:", OUTPUT_DIR)

## Reproduce the inference transform exactly

CLAHE 1.25 -> square pad -> resize 384 -> ImageNet normalisation. No augmentation, no centre crop.
This is the same path as `app/services/preprocessing_service.py`, so the metrics below describe the
deployed behaviour rather than an idealised one.

In [ ]:
class OpenCVCLAHE:
    """LAB-space CLAHE. Identical to app/services/preprocessing_service.py."""

    def __call__(self, image_rgb):
        lab = cv2.cvtColor(np.asarray(image_rgb), cv2.COLOR_RGB2LAB)
        lightness, a, b = cv2.split(lab)
        lightness = cv2.createCLAHE(clipLimit=1.25, tileGridSize=(8, 8)).apply(lightness)
        return cv2.cvtColor(cv2.merge((lightness, a, b)), cv2.COLOR_LAB2RGB)


class SquarePad:
    """Pad to square with black borders, preserving aspect ratio."""

    def __call__(self, image_rgb):
        image = np.asarray(image_rgb)
        height, width = image.shape[:2]
        side = max(height, width)
        top, left = (side - height) // 2, (side - width) // 2
        return cv2.copyMakeBorder(
            image, top, side - height - top, left, side - width - left,
            cv2.BORDER_CONSTANT, value=(0, 0, 0),
        )


val_transform = transforms.Compose([
    OpenCVCLAHE(), SquarePad(), transforms.ToPILImage(),
    transforms.Resize((INPUT_SIZE, INPUT_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])


# Split into display and tensor stages so Grad-CAM can be drawn on the exact model input.
display_transform = transforms.Compose([
    OpenCVCLAHE(), SquarePad(), transforms.ToPILImage(),
    transforms.Resize((INPUT_SIZE, INPUT_SIZE)),
])
tensor_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])


def prepare_roi(path):
    image_bgr = cv2.imread(str(path), cv2.IMREAD_COLOR)
    if image_bgr is None:
        raise RuntimeError(f"Cannot decode ROI: {path}")
    image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
    processed_rgb = np.array(display_transform(image_rgb), dtype=np.uint8, copy=True)
    return processed_rgb, tensor_transform(processed_rgb)


class ROITestDataset(Dataset):
    def __init__(self, root):
        rows = []
        for grade in range(5):
            paths = sorted((root / str(grade)).glob("*.png"))
            if not paths:
                raise RuntimeError(f"No ROI PNGs for grade {grade}: {root / str(grade)}")
            rows.extend({"path": str(path), "true_grade": grade} for path in paths)
        self.frame = pd.DataFrame(rows)

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, index):
        row = self.frame.iloc[index]
        _, tensor = prepare_roi(row.path)
        return tensor, int(row.true_grade), row.path


test_dataset = ROITestDataset(ROI_TEST_ROOT)
test_loader = DataLoader(
    test_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True,
)
print(test_dataset.frame.groupby("true_grade").size())
print("Total ROIs:", len(test_dataset))

## Load the checkpoint and evaluate once

Architecture and loss type are verified before the weights load, so a mismatched checkpoint fails
loudly here instead of producing quietly wrong numbers.

In [ ]:
class SEResNeXt50Model(nn.Module):
    """Focal CORN variant. num_classes=4 -> 4 ordinal thresholds for 5 KL grades."""
    ARCHITECTURE = "seresnext50_32x4d_linear_gradcam_ordinal"
    def __init__(self):
        super().__init__()
        self.backbone = timm.create_model(
            "seresnext50_32x4d", pretrained=PRETRAINED, num_classes=NUM_CLASSES - 1
        )
    @property
    def gradcam_target_layer(self):
        return self.backbone.layer4
    def forward(self, images):
        return self.backbone(images)
build_model = SEResNeXt50Model
checkpoint = torch.load(CHECKPOINT_PATH, map_location="cpu", weights_only=False)
if checkpoint.get("loss_type") not in (None, "focal_corn"):
    raise RuntimeError(f"Expected a Focal CORN checkpoint, got {checkpoint.get('loss_type')}")
if checkpoint.get("architecture") not in (None, build_model.ARCHITECTURE):
    raise RuntimeError(f"Unexpected architecture: {checkpoint.get('architecture')}")

model = build_model().to(DEVICE)
model.load_state_dict(checkpoint["model_state_dict"], strict=True)
model.eval()

all_paths, all_labels, all_probabilities = [], [], []
with torch.inference_mode():
    for images, labels, paths in tqdm(test_loader, desc="Locked ROI test evaluation"):
        probabilities = corn_probas(model(images.to(DEVICE, non_blocking=True)).float()).cpu().numpy()
        all_paths.extend(paths)
        all_labels.extend(labels.numpy().tolist())
        all_probabilities.extend(probabilities)

labels = np.asarray(all_labels, dtype=int)
probabilities = np.asarray(all_probabilities, dtype=float)
# Reverse the chain rule to recover pseudo-logits for corn_label_from_logits.
log_p = np.log(np.clip(probabilities, 1e-9, 1.0))
log_one_minus_p = np.log(np.clip(1.0 - probabilities, 1e-9, 1.0))
pseudo_logits = log_p - log_one_minus_p
predictions = corn_label_from_logits(torch.as_tensor(pseudo_logits)).numpy()

precision, recall, macro_f1, _ = precision_recall_fscore_support(
    labels, predictions, average="macro", zero_division=0,
)
per_class = precision_recall_fscore_support(labels, predictions, labels=range(5), zero_division=0)

metrics = {
    "samples": int(len(labels)),
    "accuracy": float(accuracy_score(labels, predictions)),
    "qwk": float(cohen_kappa_score(labels, predictions, weights="quadratic")),
    "mae": float(np.abs(labels - predictions).mean()),
    "off_by_one_accuracy": float((np.abs(labels - predictions) <= 1).mean()),
    "macro_precision": float(precision),
    "macro_recall": float(recall),
    "macro_f1": float(macro_f1),
    "macro_ap": float(average_precision_score(np.eye(5)[labels], probabilities, average="macro")),
    "macro_roc_auc_ovr": float(roc_auc_score(labels, probabilities, multi_class="ovr", average="macro")),
    "grade1_precision": float(per_class[0][1]),
    "grade1_recall": float(per_class[1][1]),
}
(OUTPUT_DIR / "test_metrics.json").write_text(json.dumps(metrics, indent=2))

prediction_frame = pd.DataFrame({
    "roi_path": all_paths,
    "true_grade": labels,
    "predicted_grade": predictions,
    "confidence": probabilities.max(axis=1),
})
for grade in range(5):
    prediction_frame[f"probability_grade_{grade}"] = probabilities[:, grade]
prediction_frame.to_csv(OUTPUT_DIR / "test_predictions.csv", index=False)

print(json.dumps(metrics, indent=2))
print()
print(classification_report(labels, predictions, digits=4, zero_division=0))

figure, axis = plt.subplots(figsize=(7, 6))
sns.heatmap(
    confusion_matrix(labels, predictions, labels=range(5)), annot=True, fmt="d",
    cmap="Blues", xticklabels=range(5), yticklabels=range(5), ax=axis,
)
axis.set(xlabel="Predicted KL grade", ylabel="True KL grade",
         title="Locked YOLO ROI Test Confusion Matrix (Focal CORN)")
figure.tight_layout()
figure.savefig(OUTPUT_DIR / "test_confusion_matrix.png", dpi=180, bbox_inches="tight")
plt.show()


## Grad-CAM review

Correct cases show what the model used when it was right. Errors get two CAMs — one for the
predicted class, one for the true class — which separates plausible adjacent-grade confusion from
shortcut evidence outside the joint.

In [ ]:
class GradCAM:
    def __init__(self, model):
        self.model = model
        self.activations = None
        self.gradients = None
        self.handle = model.gradcam_target_layer.register_forward_hook(self._capture)

    def _capture(self, module, inputs, output):
        self.activations = output
        output.register_hook(self._capture_gradient)

    def _capture_gradient(self, gradient):
        self.gradients = gradient

    def __call__(self, tensor, class_index):
        self.model.zero_grad(set_to_none=True)
        logits = self.model(tensor)
        logits[:, class_index].sum().backward()
        weights = self.gradients.mean(dim=(2, 3), keepdim=True)
        cam = F.relu((weights * self.activations).sum(dim=1, keepdim=True))
        cam = F.interpolate(
            cam, size=(INPUT_SIZE, INPUT_SIZE), mode="bilinear", align_corners=False
        )[0, 0]
        cam = cam.detach().cpu().numpy()
        return (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)

    def remove(self):
        self.handle.remove()


def cam_overlay(image_rgb, cam, alpha=0.42):
    heatmap = cv2.applyColorMap(np.uint8(255 * cam), cv2.COLORMAP_JET)
    heatmap = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB)
    return cv2.addWeighted(image_rgb, 1.0 - alpha, heatmap, alpha, 0)


def render_case(path, class_index):
    processed_rgb, tensor = prepare_roi(path)
    cam = gradcam(tensor.unsqueeze(0).to(DEVICE), class_index)
    return processed_rgb, cam_overlay(processed_rgb, cam)


gradcam = GradCAM(model)
correct_dir = OUTPUT_DIR / "gradcam_correct_by_true_grade"
incorrect_dir = OUTPUT_DIR / "gradcam_misclassified_pairs"
correct_dir.mkdir()
incorrect_dir.mkdir()

for true_grade in range(5):
    cases = prediction_frame[
        (prediction_frame.true_grade == true_grade)
        & (prediction_frame.predicted_grade == true_grade)
    ].head(CASES_PER_GRADE)
    # squeeze=False keeps axes 2-D even when CASES_PER_GRADE is 1.
    figure, axes = plt.subplots(
        2, CASES_PER_GRADE, figsize=(4 * CASES_PER_GRADE, 7), squeeze=False
    )
    for column in range(CASES_PER_GRADE):
        axes[0, column].axis("off")
        axes[1, column].axis("off")
        if column >= len(cases):
            continue
        case = cases.iloc[column]
        source, overlay = render_case(case.roi_path, true_grade)
        axes[0, column].imshow(source)
        axes[0, column].set_title(f"True G{true_grade}\n{Path(case.roi_path).name}", fontsize=9)
        axes[1, column].imshow(overlay)
        axes[1, column].set_title(f"Correct G{true_grade}, p={case.confidence:.3f}", fontsize=9)
    figure.suptitle(f"Correct Grad-CAM examples: true KL grade {true_grade}")
    figure.tight_layout()
    figure.savefig(correct_dir / f"grade_{true_grade}_correct.png", dpi=160, bbox_inches="tight")
    plt.show()
    plt.close(figure)

misclassified_rows = []
for true_grade in range(5):
    cases = prediction_frame[
        (prediction_frame.true_grade == true_grade)
        & (prediction_frame.predicted_grade != true_grade)
    ].head(CASES_PER_GRADE)
    for case_number, (_, case) in enumerate(cases.iterrows(), start=1):
        source, predicted_overlay = render_case(case.roi_path, int(case.predicted_grade))
        _, true_overlay = render_case(case.roi_path, int(case.true_grade))
        figure, axes = plt.subplots(1, 3, figsize=(14, 5))
        axes[0].imshow(source)
        axes[0].set_title(f"ROI | true G{case.true_grade}")
        axes[1].imshow(predicted_overlay)
        axes[1].set_title(f"Predicted G{case.predicted_grade} | p={case.confidence:.3f}")
        axes[2].imshow(true_overlay)
        axes[2].set_title(f"True-class G{case.true_grade} CAM")
        for axis in axes:
            axis.axis("off")
        figure.tight_layout()
        output_path = incorrect_dir / (
            f"true_g{case.true_grade}_pred_g{case.predicted_grade}"
            f"_{case_number}_{Path(case.roi_path).stem}.png"
        )
        figure.savefig(output_path, dpi=160, bbox_inches="tight")
        plt.show()
        plt.close(figure)
        misclassified_rows.append({
            "roi_path": case.roi_path,
            "true_grade": int(case.true_grade),
            "predicted_grade": int(case.predicted_grade),
            "confidence": float(case.confidence),
            "cam_pair_path": str(output_path),
        })

pd.DataFrame(misclassified_rows).to_csv(OUTPUT_DIR / "gradcam_misclassified_index.csv", index=False)
gradcam.remove()
print("Correct CAM grids:      ", correct_dir)
print("Misclassified CAM pairs:", incorrect_dir)

## Emit a report row

`report_row.json` mirrors the column names in `docs/report/report.csv`, so the record can be filled
in from measured output instead of retyped by hand.

In [ ]:
report_row = {
    "model_family": "SE-ResNeXt-50 32x4d",
    "run_timestamp": RUN_TIMESTAMP,
    "timezone": "UTC",
    "record_type": "evaluation-only",
    "status": "completed",
    "source_report": "docs/report/paper.md",
    "notebook_archive": "notebooks/seresnext50_32x4d/pipeline/03_evaluate_roi_test.ipynb",
    "checkpoint_directory": str(CHECKPOINT_PATH.parent.name),
    "architecture": build_model.ARCHITECTURE,
    "input_size": f"{INPUT_SIZE}x{INPUT_SIZE}",
    "roi_and_crop_policy": "YOLO square ROI expanded by 1.15 * max(box width, box height)",
    "image_processing_full": (
        f"LAB CLAHE 1.25 -> square pad -> resize {INPUT_SIZE}x{INPUT_SIZE} -> ImageNet normalization"
    ),
    "loss_function": "Focal CORN (gamma=2.0, alpha=0.25)",
    "dataset_split": "Locked YOLO-ROI test split",
    "test_n": metrics["samples"],
    "test_accuracy": round(metrics["accuracy"], 4),
    "test_qwk": round(metrics["qwk"], 4),
    "test_mae": round(metrics["mae"], 4),
    "test_macro_precision": round(metrics["macro_precision"], 4),
    "test_macro_recall": round(metrics["macro_recall"], 4),
    "test_macro_f1": round(metrics["macro_f1"], 4),
    "test_grade1_precision": round(metrics["grade1_precision"], 4),
    "test_grade1_recall": round(metrics["grade1_recall"], 4),
    "test_average_precision": round(metrics["macro_ap"], 4),
    "test_roc_auc": round(metrics["macro_roc_auc_ovr"], 4),
}
(OUTPUT_DIR / "report_row.json").write_text(json.dumps(report_row, indent=2))

manifest = {
    "checkpoint_path": str(CHECKPOINT_PATH),
    "test_roi_root": str(ROI_TEST_ROOT),
    "preprocessing": "CLAHE(1.25) -> square pad -> resize(384) -> ImageNet normalization",
    "evaluation_only": True,
    "metrics": metrics,
}
(OUTPUT_DIR / "evaluation_manifest.json").write_text(json.dumps(manifest, indent=2))

print(json.dumps(report_row, indent=2))
print()
print("Evidence package:", OUTPUT_DIR)
print("Promote to .env only after reviewing the Grad-CAM grids above.")